In [ ]:
import pandas as pd

raw = pd.read_csv('events.csv')

In [ ]:
with open('events.csv', 'r', encoding='utf-8') as f:
    contenido = f.read()

# Buscar saltos de línea raros
print(repr(contenido[:2000]))

In [ ]:
import pandas as pd
import io
import re

# Leer archivo como texto
with open('events.csv', 'r', encoding='utf-8') as f:
    contenido = f.read()

# Reparar: si una línea NO empieza con "Estrella" o "Team" 
# y NO está vacía, la pegamos a la línea anterior
lineas = contenido.split('\n')
lineas_limpias = []

for linea in lineas:
    if linea.startswith('Team') or linea.startswith('Estrella') or linea.startswith('Rival') or linea == '':
        lineas_limpias.append(linea)
    else:
        # Esta línea es continuación de la anterior
        if lineas_limpias:
            lineas_limpias[-1] = lineas_limpias[-1] + linea

contenido_limpio = '\n'.join(lineas_limpias)

# Leer desde string
df = pd.read_csv(io.StringIO(contenido_limpio))

# Filtrar solo Estrella
df = df[df['Team'] == 'Estrella'].copy()

print(df.head(10))
print("\nTotal filas:", len(df))
print("\nEventos:")
print(df['Event'].value_counts())

In [ ]:
df['Event'].value_counts()
df['Player'].unique()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import io

# Estilo visual para matplotlib
plt.style.use('dark_background')

In [ ]:
# 1. Leer el archivo crudo como texto para arreglar los saltos de línea de FC Python
with open('events.csv', 'r', encoding='utf-8') as f:
    contenido = f.read()

lineas = contenido.split('\n')
lineas_limpias = []

for linea in lineas:
    # Si la línea empieza con Team o Estrella, es una fila nueva
    if linea.startswith('Team') or linea.startswith('Estrella') or linea == '':
        lineas_limpias.append(linea)
    else:
        # Si no, es un evento cortado: lo pegamos a la fila anterior
        if lineas_limpias:
            lineas_limpias[-1] = lineas_limpias[-1] + linea

contenido_limpio = '\n'.join(lineas_limpias)

# 2. Cargar a Pandas
df = pd.read_csv(io.StringIO(contenido_limpio))

# 3. Filtrar solo nuestro equipo
df = df[df['Team'] == 'Estrella'].copy()

# 4. Convertir coordenadas a numérico y eliminar filas sin evento
for col in ['X', 'Y']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['X', 'Y', 'Event'])
df = df.sort_values(by=['Mins', 'Secs']).reset_index(drop=True)

print(f"Dataset limpio: {len(df)} eventos válidos.")
print("Jugadores:", df['Player'].nunique())

In [ ]:
# Calculamos la posición promedio de Franco (Arquero)
franco_mean_x = df[df['Player'] == 'Franco']['X'].mean()
print(f"Posición promedio de Franco en X: {franco_mean_x:.1f}")

# Si Franco está a la derecha (X > 50), rotamos todo 180° para atacar de izquierda a derecha
if franco_mean_x > 50:
    print("➡️ Rotando coordenadas para atacar de izquierda a derecha...")
    df['X'] = 100 - df['X']
    df['Y'] = 100 - df['Y']
else:
    print("⬅️ Orientación correcta. No se rota.")

In [ ]:
passes = []

for i in range(len(df) - 1):
    row = df.iloc[i]
    next_row = df.iloc[i + 1]

    # Lógica: un 'Pase' seguido inmediatamente por una 'Recepción' del mismo equipo
    if (row['Event'] == 'Pase' and 
        next_row['Event'] == 'Recepción' and 
        row['Team'] == next_row['Team']):
        
        passes.append({
            'passer': row['Player'],
            'receiver': next_row['Player'],
            'x': row['X'],
            'y': row['Y'],
            'end_x': next_row['X'],
            'end_y': next_row['Y']
        })

passes_df = pd.DataFrame(passes)
print(f"✅ Pases completados detectados: {len(passes_df)}")

In [ ]:
players = df['Player'].unique()
avg_list = []

for player in players:
    player_df = df[df['Player'] == player]
    
    avg_list.append({
        'Player': player,
        'X': player_df['X'].mean(),
        'Y': player_df['Y'].mean(),
        'touches': len(player_df)  # Cantidad total de intervenciones
    })

avg_positions = pd.DataFrame(avg_list)

In [ ]:
# Contamos cuántos pases hubo entre cada par de jugadores
connections = passes_df.groupby(['passer', 'receiver']).size().reset_index(name='count')

# Mostramos las conexiones más fuertes
print("Top conexiones:")
display(connections.sort_values(by='count', ascending=False).head(10))

In [ ]:
# Configuramos la cancha tipo OPTA (0-100) con estilo oscuro
pitch = Pitch(
    pitch_type='opta',
    pitch_color='#22312b',
    line_color='white',
    line_alpha=0.3
)

fig, ax = pitch.draw(figsize=(14, 9))

# 1. DIBUJAR LÍNEAS (PASES)
for _, row in connections.iterrows():
    p1 = avg_positions[avg_positions['Player'] == row['passer']]
    p2 = avg_positions[avg_positions['Player'] == row['receiver']]
    
    if p1.empty or p2.empty:
        continue
        
    x1, y1 = p1[['X','Y']].values[0]
    x2, y2 = p2[['X','Y']].values[0]
    
    # El grosor de la línea aumenta según la cantidad de pases
    pitch.lines(
        x1, y1, x2, y2,
        lw=1 + row['count'] * 0.7,  
        color='#ffffff',
        alpha=0.5,
        zorder=1,
        ax=ax
    )

# 2. DIBUJAR NODOS (JUGADORES)
for _, row in avg_positions.iterrows():
    x, y = row['X'], row['Y']
    touches = row['touches']
    player = row['Player']
    
    # Tamaño del nodo: base 300 + 15 por cada intervención (para que los que tocan poco se vean)
    node_size = 300 + (touches * 15)
    
    # Círculo del jugador
    pitch.scatter(
        x, y,
        s=node_size,
        color='#00ff9c',          # Verde neón profesional
        edgecolors='#000000',
        linewidth=1.5,
        zorder=2,
        ax=ax
    )
    
    # Nombre centrado dentro del nodo
    ax.text(
        x, y,
        player,
        ha='center',
        va='center',
        fontsize=9,
        color='black',
        fontweight='bold',
        zorder=3
    )

# 3. TÍTULO Y ESTILO FINAL
ax.set_title(
    'Red de Pases - Estrella (1er Tiempo) | 7ma División',
    fontsize=16,
    color='white',
    pad=20,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# Filtrar solo eventos de juego (no recuperaciones, goles, etc.)
juego_df = df[df['Event'].isin(['Pase', 'Recepción', 'Conducción'])]

pitch = Pitch(
    pitch_type='opta',
    pitch_color='#22312b',
    line_color='white',
    line_alpha=0.3
)

fig, ax = pitch.draw(figsize=(14, 9))

pitch.kdeplot(
    juego_df['X'], 
    juego_df['Y'],
    ax=ax,
    cmap='hot',      # Otro esquema de colores (más profesional)
    fill=True,
    alpha=0.7,
    n_levels=20,
    edgecolor='none'
)

ax.set_title(
    'Mapa de Calor - Pases y Recepciones',
    fontsize=16,
    color='white',
    pad=20,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# Elegí un jugador para ver su zona de influencia
jugador = 'Fran'  # Cambialo por quien quieras

jugador_df = df[df['Player'] == jugador]

pitch = Pitch(
    pitch_type='opta',
    pitch_color='#22312b',
    line_color='white',
    line_alpha=0.3
)

fig, ax = pitch.draw(figsize=(14, 9))

pitch.kdeplot(
    jugador_df['X'], 
    jugador_df['Y'],
    ax=ax,
    cmap='hot',       # Azul-verde (más elegante)
    fill=True,
    alpha=0.7,
    n_levels=15,
    edgecolor='none'
)

# Marcar posición promedio
x_avg = jugador_df['X'].mean()
y_avg = jugador_df['Y'].mean()

pitch.scatter(
    x_avg, y_avg,
    s=400,
    color='red',
    edgecolors='white',
    linewidth=2,
    ax=ax,
    zorder=5
)

ax.text(
    x_avg, y_avg,
    jugador,
    ha='center',
    va='center',
    fontsize=12,
    color='white',
    fontweight='bold',
    zorder=6
)

ax.set_title(
    f'Mapa de Calor - {jugador}',
    fontsize=16,
    color='white',
    pad=20,
    fontweight='bold'
)

plt.tight_layout()
plt.show()